# Welcome, and a price you can't trust

**Lecture 1 · Build** · Géron, Chapters 1–2

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** You are not expected to type the code. You are
expected to *read* it before you run it, and to be able to say what every line
does and what would break if it changed. Cells marked **⚠ read before running**
contain a defect on purpose.

Run the cells in order. Anything that takes more than a few seconds says so.

**About the prompt boxes.** Every code cell in this notebook is preceded by a
quoted prompt, and three lines follow it: what the prompt leaves open, the
version a student typically writes instead, and how you would catch a wrong
answer. Those three lines are the part worth reading twice.

The prompts here are **specifications, not transcripts** — this is what you
would have to ask for in order to get this cell, not a recording of somebody
asking for it. If your own prompt is vaguer than the box, expect worse code than
the cell below it.

*Lecture 19 is the one exception in this course.* It was built cell by cell
against Colab's Gemini 3.1 Pro, and its prompts are verbatim. It says so itself.


## 1 · Setup

> **Prompt · setup**
>
> **input** · nothing
>
> **output** · the version of every library this notebook depends on, and one seed
>
> **constraint** · ASSERT the scikit-learn version rather than printing it — `root_mean_squared_error` arrived in 1.4, and on an older Colab image the failure is an ImportError twenty cells from here

**Watch this prompt.**

* **Left open:** that RANDOM_STATE is defined once and used for every split, every model and every shuffle. A notebook with three different seeds in it cannot be reproduced by reading it.
* **The usual student version:** printing the versions and not checking them, so the notebook reports its own incompatibility as information rather than as an error.
* **How you would catch it:** not examinable, and it is here because a version mismatch produces a confusing error in a cell that has nothing to do with versions.

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: this is engineering hygiene, not machine learning. It is here
# because a version mismatch produces a confusing error twenty cells later.
import sys, sklearn, numpy as np, pandas as pd, matplotlib

print(f"python       {sys.version.split()[0]}")
print(f"scikit-learn {sklearn.__version__}")
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")

# root_mean_squared_error arrived in scikit-learn 1.4
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 4), \
    "This notebook needs scikit-learn >= 1.4.  In Colab: %pip install -U scikit-learn"

RANDOM_STATE = 42          # every split, every model, every shuffle
pd.set_option("display.width", 100)

## 2 · The data

> **Prompt · the data**
>
> **input** · the California housing tarball
>
> **output** · 20,640 districts and 10 columns
>
> **constraint** · a FUNCTION that downloads if absent and reads if present — the data will change, and you will need this on another machine
>
> **check** · assert the shape, rather than trusting the download

**Watch this prompt.**

* **Left open:** what to do if the download is truncated. A short read gives a smaller frame and the assert catches it; anything subtler it will not.
* **The usual student version:** downloading by hand and reading a path under ~/Downloads. It works on your machine and nowhere else, which you discover at the demo.
* **How you would catch it:** delete `datasets/` and re-run. If the cell cannot rebuild its own input from nothing, it is not reproducible, it is cached.

In [ ]:
# --- the data ----------------------------------------------------------------
# A function, not a manual download: the data will change, and you will need
# this on another machine.  ~5 s the first time, instant afterwards.
from pathlib import Path
import tarfile, urllib.request

def load_housing():
    tarball = Path("datasets/housing.tgz")
    if not tarball.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path="datasets", filter="data")
    return pd.read_csv("datasets/housing/housing.csv")

housing_full = load_housing()

assert housing_full.shape == (20640, 10), f"unexpected shape {housing_full.shape}"
print(f"{len(housing_full):,} districts, {housing_full.shape[1]} columns")
housing_full.head()

### What is in it

Ten attributes per district. One of them is not numeric, and one column has
holes in it. Find both before reading on.

> **Prompt · what is in it**
>
> **input** · the loaded frame
>
> **output** · every column, its type and its non-null count
>
> **constraint** · `.info()`, not `.head()` — the two things worth finding here are a non-numeric column and a column with holes in it, and neither is visible in five rows

**Watch this prompt.**

* **Left open:** which column is which. Find both before reading on; the next cell names them.
* **The usual student version:** `.head()` and an impression. A column that is 99% present looks complete in the first five rows, and the dtype of a mostly-numeric-looking column is not visible at all.
* **How you would catch it:** non-null counts against the row count. That subtraction is the missing-value audit, and it is free.

In [ ]:
housing_full.info()

> **Prompt · count the holes, and the categories**
>
> **input** · the frame
>
> **output** · how many districts are missing total_bedrooms, and the counts of every category level
>
> **constraint** · print the missing count as a PERCENTAGE as well as a count — 207 sounds like a lot and 1% does not

**Watch this prompt.**

* **Left open:** that ISLAND has five districts in the whole of California. Remember it: it comes back in the next lecture and it does not announce itself when it breaks.
* **The usual student version:** dropping the rows with missing bedrooms, which throws away 207 districts to avoid writing one imputer.
* **How you would catch it:** `value_counts()` on every categorical, always. A level with n=5 is a level that will be absent from some cross-validation folds.

In [ ]:
n_missing = housing_full["total_bedrooms"].isna().sum()
print(f"total_bedrooms is missing in {n_missing} districts "
      f"({100 * n_missing / len(housing_full):.1f}%)")
print()
print(housing_full["ocean_proximity"].value_counts())

`ISLAND` has five districts in the whole of California. Remember that; it comes
back in the next lecture and it does not announce itself when it breaks.

## 3 · Split before you look

This is the first rule and the easiest one to break. Everything you learn from
the data *before* the split leaks into the choices you make afterwards — through
you, not through the code. There is no library that prevents this.

We stratify on income because the experts told us income predicts price. A
random split gets the income mix wrong by up to 6.4%; stratifying gets it wrong
by 0.36%.

> **Prompt · split before you look**
>
> **input** · the whole frame
>
> **output** · a stratified 80/20 split
>
> **constraint** · stratify on the INCOME BAND, because the experts said income predicts price — a random split gets the income mix wrong by up to 6.4% and stratifying gets it wrong by 0.36%
>
> **check** · assert the two halves sum to the whole and that their indices are disjoint

**Watch this prompt.**

* **Left open:** why this is the first rule and the easiest to break. Everything you learn from the data BEFORE the split leaks into the choices you make afterwards — through you, not through the code. There is no library that prevents this.
* **The usual student version:** exploring first and splitting later, because exploring is the interesting part. By then you have chosen which features to engineer using the test rows.
* **How you would catch it:** the comment on the last line — from here to the final cell, `test_set` is not touched again. Write it down, in the code, where it will be read.

In [ ]:
from sklearn.model_selection import train_test_split

income_cat = pd.cut(housing_full["median_income"],
                    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                    labels=[1, 2, 3, 4, 5])

train_set, test_set = train_test_split(
    housing_full, test_size=0.2, random_state=RANDOM_STATE, stratify=income_cat)

# assert, do not hope
assert len(train_set) + len(test_set) == len(housing_full)
assert set(train_set.index).isdisjoint(test_set.index), "the split overlaps"
print(f"train {len(train_set):,}   test {len(test_set):,}")

# From here to the very last cell, `test_set` is not touched again.
housing = train_set.copy()

## 4 · Look — at the training set only

Two things should jump out of the histograms. Take thirty seconds before you
scroll.

> **Prompt · look — at the TRAINING set only**
>
> **input** · the training half
>
> **output** · a histogram of every numeric column
>
> **constraint** · `housing`, the training copy, not `housing_full` — the whole point of the previous cell was to make this cell safe

**Watch this prompt.**

* **Left open:** two things that should jump out. Take thirty seconds before scrolling: the income is not in dollars, and the TARGET is capped.
* **The usual student version:** plotting the full frame out of habit. It is one word different and it undoes the split.
* **How you would catch it:** 50 bins, not the default 10. A cap at the top of a distribution is one bar, and at 10 bins it is inside a bar with everything else.

In [ ]:
import matplotlib.pyplot as plt

housing.hist(bins=50, figsize=(12, 8))
plt.tight_layout(); plt.show()

**The income is not in dollars** — it is scaled, and capped at 15.0001.

**The target is capped too**, and the target is our label. Count it rather than
squinting at it:

> **Prompt · count the cap rather than squinting at it**
>
> **input** · the target column
>
> **output** · how many districts sit at the cap, and the commonest values below it
>
> **constraint** · count it — a histogram shows you a spike and a count tells you whether it is 5% of your labels or 0.5%

**Watch this prompt.**

* **Left open:** what the commonest values have in common. Every one of them is a multiple of $12,500 — artefacts of how the survey recorded prices, not facts about California.
* **The usual student version:** noticing the cap and moving on. The target is the LABEL, so a cap on it means 5% of your training rows have a label that is not the answer, and no model can be right about them.
* **How you would catch it:** `value_counts()` on a continuous target should be almost flat. Where it is not, the recording process is visible, and that is a fact about the survey rather than the world.

In [ ]:
capped = (housing["median_house_value"] >= 500_000).sum()
print(f"{capped} districts sit at the cap "
      f"({100 * capped / len(housing):.1f}% of the training set)")

# which values do districts actually pile up on?
counts = housing["median_house_value"].value_counts()
print(f"\na typical price is shared by {counts.median():.0f} districts")
print("\nthe five commonest values below the cap:")
print(counts.drop(counts.index.max()).head(5))

Every one of those is a multiple of **$12,500**. They are artefacts of how the
survey recorded prices, not facts about California.

A well-known description of this dataset names fainter lines at \$450,000,
\$350,000 and \$280,000. Check that claim against the counts above before you
believe it — one of the three is real, one is marginal, and one is
indistinguishable from the background.

> **Prompt · check the famous claim**
>
> **input** · three values named in a well-known description of this dataset
>
> **output** · how many districts sit at each
>
> **constraint** · check the claim against the counts you just computed rather than repeating it

**Watch this prompt.**

* **Left open:** the answer: one of the three is real, one is marginal, and one is indistinguishable from the background. The cell does not say which.
* **The usual student version:** repeating 'there are also lines at 450,000, 350,000 and 280,000' because it is in the book. Two of the three do not survive a count.
* **How you would catch it:** when a source names specific numbers about your data, the numbers are checkable. Three lines, and you either confirm it or you have found something.

In [ ]:
for value in (450_000, 350_000, 280_000):
    print(f"${value:>9,}  {counts.get(value, 0):>4d} districts")

## 5 · A number to compare against

Rule 2 of this course: *a metric with nothing to compare it to is decoration.*

So before building anything, measure the dumbest possible model — predict the
same number for every district. Everything you build today has to beat this, and
by how much is the only thing that will make your RMSE mean anything.

> **Prompt · a number to compare against**
>
> **input** · the training mean
>
> **output** · the RMSE of predicting it for every district, and what the human experts cost
>
> **constraint** · compute the dumbest possible model BEFORE building anything — everything today has to beat it, and by how much is the only thing that will make your RMSE mean anything

**Watch this prompt.**

* **Left open:** that the expert figure is quoted, not measured. About 30% off on a typical $200,000 district, which the notebook converts to dollars so the two numbers are on one scale.
* **The usual student version:** reporting an RMSE of $68,000 with nothing beside it. Is that good? The question is unanswerable without this cell.
* **How you would catch it:** rule 2 of this course: a metric with nothing to compare it to is decoration. This is the cheapest possible comparison and it takes four lines.

In [ ]:
from sklearn.metrics import root_mean_squared_error

y_train = housing["median_house_value"]
y_test  = test_set["median_house_value"]

baseline = np.full(len(y_test), y_train.mean())
baseline_rmse = root_mean_squared_error(y_test, baseline)
print(f"predict the training mean  ->  RMSE ${baseline_rmse:,.0f}")
print(f"the human experts are off by about 30%, i.e. roughly  ${0.30 * 200_000:,.0f}")

## 6 · Commit

**Stop. On paper, now.** Not in this notebook — on paper, where you cannot
quietly revise it.

```
Metric:                                        ____________
Target RMSE for a good system:               $ ____________
RMSE I expect from the model I build today:  $ ____________
```

A prediction you can silently revise is not a prediction.

## 7 · The same request, twice

Two prompts for the same job. The first is what most people type. The second is
the same request with the one under-specified place closed.

> **Weak** · *"Load the housing data, scale the features and split it into
> training and test sets."*

> **Usable** · *"Split it 80/20 first, with a fixed seed. **Then** fit the
> imputer and the scaler on the training half only and apply them to both. No
> statistic computed on the test rows may touch the training path."*

Both are answerable. Below is the code the **weak** one returns.
**⚠ Read before running.** It runs, it imports nothing exotic, and it prints a
believable number.

> **Prompt · ⚠ what the weak prompt returns**
>
> **input** · 'load the housing data, scale the features and split it into training and test sets'
>
> **output** · a fitted linear model and its RMSE
>
> **constraint** · run it exactly as returned — it imports nothing exotic and prints a believable number

**Watch this prompt.**

* **Left open:** the order. The request named scaling and splitting and did not say which comes first, and one of the two orders is a leak.
* **The usual student version:** this exact code. It is what the weak prompt returns, and nothing in the output flags it.
* **How you would catch it:** reviewer question 1 — what touched the test set? `fit_transform` ran on ALL the rows, so the median that fills the missing values and the mean and standard deviation that scale every column were computed from a set that includes the rows we then call the test set.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X_all = housing_full.select_dtypes(include=[np.number]).drop(
    columns=["median_house_value"])
y_all = housing_full["median_house_value"]

prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
X_scaled = prep.fit_transform(X_all)          # <-- all 20,640 rows

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_all, test_size=0.2, random_state=RANDOM_STATE)

leaky = LinearRegression().fit(X_tr, y_tr)
print(f"RMSE ${root_mean_squared_error(y_te, leaky.predict(X_te)):,.0f}   looks fine")

### Reviewer question 1: what touched the test set?

`fit_transform` ran on **all** the rows. The median that fills the missing
values, and the mean and standard deviation that scale every column, were all
computed from a set that includes the rows we then call the test set.

So the model is evaluated on rows whose own values helped define the
transformation applied to them.

**Now price the two prompts against each other** — do not guess:

> **Prompt · what the weak prompt cost**
>
> **input** · the same rows and the same model, once under each of the two procedures
>
> **output** · the test RMSE of each, and the difference between them
>
> **constraint** · change ONE thing — the order of the split and the fit — so the difference is attributable to the prompt and to nothing else
>
> **check** · twenty random splits, not one, and print the split-to-split spread of each RMSE beside the difference; a difference smaller than that spread has not been measured, it has been sampled

**Watch this prompt.**

* **Left open:** what counts as a difference worth reporting. The cell prints the spread next to it so you can answer that for yourself.
* **The usual student version:** subtracting two numbers from a single split and quoting the result. On one split of this data the weak procedure can look either better or worse.
* **How you would catch it:** count the sign. A quantity whose sign is not stable across splits is a draw from a distribution, not a cost.

In [ ]:
# ~1 s: twenty splits, two procedures, one model.
from sklearn.model_selection import ShuffleSplit

# what the WEAK prompt returned: scale everything once, split afterwards
X_leaked = make_pipeline(SimpleImputer(strategy="median"),
                         StandardScaler()).fit_transform(X_all)

weak, usable = [], []
for tr, te in ShuffleSplit(n_splits=20, test_size=0.2,
                           random_state=RANDOM_STATE).split(X_all):
    model = LinearRegression().fit(X_leaked[tr], y_all.iloc[tr])
    weak.append(root_mean_squared_error(y_all.iloc[te], model.predict(X_leaked[te])))

    # what the USABLE prompt returned: split first, fit the preprocessing on train only
    prep_ok = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    model = LinearRegression().fit(prep_ok.fit_transform(X_all.iloc[tr]), y_all.iloc[tr])
    usable.append(root_mean_squared_error(y_all.iloc[te],
                                          model.predict(prep_ok.transform(X_all.iloc[te]))))

weak, usable = np.array(weak), np.array(usable)
cost = weak - usable

print(f"weak prompt    RMSE ${weak.mean():,.0f}  (spread over splits ±${weak.std():,.0f})")
print(f"usable prompt  RMSE ${usable.mean():,.0f}  (spread over splits ±${usable.std():,.0f})")
print(f"\nwhat the weak prompt cost  ${cost.mean():,.2f}  (±${cost.std():,.2f}, "
      f"from ${cost.min():,.2f} to ${cost.max():,.2f})")
print(f"the weak prompt was worse on {(cost > 0).sum()} of the {len(cost)} splits, "
      f"better on {(cost < 0).sum()}, identical on {(cost == 0).sum()}")

### So the weak prompt cost nothing here

Read the last two lines together. The cost is orders of magnitude below the
spread of the numbers it is a difference of, and its sign changes from one split
to the next. Quote the single-split figure as a finding and you have quoted
noise.

The rule is procedural all the same: **split first** — not because the cost is
always this size, but because nothing in either output tells you what size it is.

You will meet the same error, from the same kind of weak prompt, worth far more
than this in about an hour.

## 8 · Build it properly

One `Pipeline`, so that cross-validation refits *all* of it on each fold and the
leak becomes structurally impossible rather than merely avoided.

> **Prompt · build it properly**
>
> **input** · the training features
>
> **output** · one ColumnTransformer handling numeric and categorical columns
>
> **constraint** · one Pipeline, so cross-validation refits ALL of it on each fold and the leak becomes structurally impossible rather than merely avoided
>
> **check** · assert the numeric and categorical column lists together account for every column — a column silently dropped here is a feature you never notice you are not using

**Watch this prompt.**

* **Left open:** `handle_unknown='ignore'` on the encoder. It is the right choice and it is also how ISLAND becomes an all-zero row with no warning, two sections into the next lecture.
* **The usual student version:** listing the numeric columns by hand and forgetting one. The set assert is what catches it.
* **How you would catch it:** the difference between avoided and impossible. A pipeline does not make you more careful; it removes the option.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

X_train = housing.drop(columns=["median_house_value"])

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = ["ocean_proximity"]

preprocessing = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

assert set(num_cols) | set(cat_cols) == set(X_train.columns), "a column was dropped"
print(f"{len(num_cols)} numeric + {len(cat_cols)} categorical")

> **Prompt · ⏱ 20 s — three models, scored on their own training data**
>
> **input** · the three model families
>
> **output** · each one's RMSE on the rows it was fitted to
>
> **constraint** · score on the TRAINING data, deliberately — this is the setup for the next lecture and not a result

**Watch this prompt.**

* **Left open:** that one of the three numbers is zero, and that two of the three are meaningless. The notebook does not say which.
* **The usual student version:** reporting these. A tree with no depth limit puts every training row in its own leaf, and its zero is not a model that is perfect, it is a model that has memorised.
* **How you would catch it:** write your best RMSE on paper next to what you predicted, and do not fix anything. Being wrong is the point and the diagnosis is the next ninety minutes.

In [ ]:
# ~20 s: the forest is 100 trees on 16,512 rows.
models = {
    "Linear regression": LinearRegression(),
    "Decision tree":     DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random forest":     RandomForestRegressor(n_estimators=100,
                                               random_state=RANDOM_STATE, n_jobs=-1),
}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocessing), ("model", model)]).fit(X_train, y_train)
    rmse = root_mean_squared_error(y_train, pipe.predict(X_train))
    print(f"{name:20s} RMSE on training data  ${rmse:>10,.0f}")

## 9 · Where we are

Three numbers. One of them is zero.

Write your **best RMSE** on the same sheet of paper, next to what you predicted.
Bring it to the next lecture — we open by comparing them, and two of these
numbers are meaningless.

Do not fix anything yet. Being wrong is the point, and the diagnosis is the next
ninety minutes.